# Cross-sectional momentum + time-series trend filter on a halal universe

This backtests a monthly-rebalanced, long-only strategy over the tickers in `equities-list.csv`
(rows with `Asset Type == Stock`; the ETFs in that file are excluded since the strategy ranks
individual names against each other). The CSV labels these as halal *candidates* only —
independently verify current Shariah compliance before investing.

**Strategy specification**

```
                 HALAL UNIVERSE
                       |
                       v
              12-1 momentum rank
                       |
                       v
               Top 30% stocks
                       |
                       v
              Price > 200 SMA?
                  /          \
                YES           NO
                 |             |
                 v             v
             BUY/CANDIDATE    CASH

                       |
                       v
           Buy the top 15 candidates
```

1. Rank the universe by 12-1 momentum (trailing 12-month return, most recent month skipped —
   same signal as the pure cross-sectional momentum notebook).
2. Narrow to the top 30% of that ranking.
3. Within that shortlist, keep only names trading above their own 200-day simple moving average
   — a time-series trend filter meant to screen out names that only rank high on momentum because
   they bounced sharply off a large decline, not because they're in a genuine uptrend.
4. Buy the top 15 names (by momentum rank) that survive both filters, equal-weighted. If fewer
   than 15 names pass, the unfilled slots are simply left in cash rather than concentrated into
   the survivors — this strategy can be partially or fully in cash, unlike the pure momentum
   notebook, which is always fully invested.
5. Rebalance monthly: the whole ranking/filtering/selection process runs again every month end.

Daily bars are reused from the pure cross-sectional momentum notebook's cache (`data/daily/`) —
the acquisition cell below is identical and simply fetches anything that isn't already cached.

## Before running

1. Install dependencies: `pip install ib-insync pandas numpy matplotlib pandas-datareader`
2. Run IBKR Trader Workstation or IB Gateway, enable API connections, and use a market-data-enabled
   account.
3. Set the host, port, and client ID below. Typical paper ports are 7497 for TWS and 4002 for
   Gateway; live ports are commonly 7496 and 4001.
4. Run the acquisition cell once. It shares the same `data/daily/{TICKER}.csv` cache as the pure
   momentum notebook, so if you've already run that notebook's download, this one has nothing left
   to fetch. Otherwise budget ~20-25 minutes for IBKR's historical-data pacing limits.

In [ ]:
from pathlib import Path
import csv
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ib_insync import IB, Stock, util
except ImportError as exc:
    raise ImportError('Install ib-insync first: pip install ib-insync') from exc

EQUITIES_LIST_PATH = Path('equities-list.csv')
DATA_DIR = Path('data') / 'daily'
LOG_PATH = Path('data') / 'daily_download_log.csv'

with open(EQUITIES_LIST_PATH, newline='') as f:
    equities_rows = list(csv.DictReader(f))
UNIVERSE = [row['Ticker'] for row in equities_rows if row['Asset Type'] == 'Stock']
print(f'{len(UNIVERSE)} candidate halal stocks in the universe')

EXCHANGE = 'SMART'
CURRENCY = 'USD'
START_DATE = pd.Timestamp('2009-01-01', tz='UTC')
END_DATE = pd.Timestamp.now(tz='UTC')
BAR_SIZE = '1 day'
DURATION = '20 Y'
WHAT_TO_SHOW = 'TRADES'
USE_RTH = True

IB_HOST = '172.27.224.1'  # '127.0.0.1'
IB_PORT = 7497
IB_CLIENT_ID = 32
PACING_SLEEP_SECONDS = 10.0

# Strategy parameters
LOOKBACK_MONTHS = 12       # momentum formation window, in months
SKIP_MONTHS = 1            # skip the most recent month (12-1 momentum)
MOMENTUM_TOP_PCT = 0.30    # "Top 30% stocks" box
SMA_WINDOW_DAYS = 200      # trend filter window, in trading days
TOP_N = 15                 # "Buy the top 15 stocks"
MIN_UNIVERSE = 20          # minimum ranked names required before trading a given month

INVESTMENT_2026_DOLLARS = 150.0
TARGET_CPI = None  # leave None to use the latest available CPI reading from FRED

## Download or load the daily bars

Identical acquisition logic to the pure cross-sectional momentum notebook: one IBKR historical
data request per ticker (daily bars support multi-year durations in a single call), cached to
`data/daily/{TICKER}.csv`, skipping anything already cached and logged `ok` in
`data/daily_download_log.csv`.

In [ ]:
def normalize_bars(bars):
    frame = util.df(bars)
    if frame is None or frame.empty:
        return pd.DataFrame()
    frame['date'] = pd.to_datetime(frame['date'], utc=True)
    frame = frame.set_index('date').sort_index()
    keep = ['open', 'high', 'low', 'close', 'volume', 'average', 'barCount']
    return frame[[column for column in keep if column in frame.columns]]

def load_cached_bars(ticker):
    frame = pd.read_csv(DATA_DIR / f'{ticker}.csv', parse_dates=['date'], index_col='date')
    frame.index = pd.DatetimeIndex(frame.index).tz_convert('UTC')
    return frame.sort_index()

def download_universe():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    util.patchAsyncio()
    ib = IB()
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID, readonly=True, timeout=20)

    log_rows = list(csv.DictReader(open(LOG_PATH))) if LOG_PATH.exists() else []
    done = {row['ticker'] for row in log_rows if row['status'] == 'ok'}

    for i, ticker in enumerate(UNIVERSE, 1):
        cache_path = DATA_DIR / f'{ticker}.csv'
        if ticker in done and cache_path.exists():
            continue
        status, n_bars, first_date, last_date, error = 'ok', 0, '', '', ''
        try:
            contract = Stock(ticker, EXCHANGE, CURRENCY)
            qualified = ib.qualifyContracts(contract)
            if not qualified or contract.conId == 0:
                raise RuntimeError('contract could not be qualified')
            bars = ib.reqHistoricalData(
                contract, endDateTime='', durationStr=DURATION,
                barSizeSetting=BAR_SIZE, whatToShow=WHAT_TO_SHOW,
                useRTH=USE_RTH, formatDate=2, keepUpToDate=False,
            )
            frame = normalize_bars(bars)
            if frame.empty:
                status = 'no_data'
            else:
                frame.to_csv(cache_path, index_label='date')
                n_bars = len(frame)
                first_date, last_date = str(frame.index.min().date()), str(frame.index.max().date())
        except Exception as exc:
            status, error = 'error', str(exc)
        print(f'[{i}/{len(UNIVERSE)}] {ticker}: {status} n_bars={n_bars} {first_date}..{last_date} {error}')
        log_rows = [r for r in log_rows if r['ticker'] != ticker]
        log_rows.append({'ticker': ticker, 'status': status, 'n_bars': n_bars,
                          'first_date': first_date, 'last_date': last_date, 'error': error})
        with open(LOG_PATH, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['ticker', 'status', 'n_bars', 'first_date', 'last_date', 'error'])
            writer.writeheader()
            writer.writerows(log_rows)
        time.sleep(PACING_SLEEP_SECONDS)

    ib.disconnect()

missing = [t for t in UNIVERSE if not (DATA_DIR / f'{t}.csv').exists()]
if missing:
    print(f'{len(missing)} tickers not yet cached, downloading from IBKR...')
    download_universe()
else:
    print('All tickers already cached.')

download_log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
if not download_log.empty:
    display(download_log['status'].value_counts().to_frame('count'))
    failed = download_log[download_log['status'] != 'ok']
    if not failed.empty:
        print('Tickers without usable data (excluded from the backtest):')
        display(failed[['ticker', 'status', 'error']])

## Build the daily and monthly price panels

The daily panel is needed for the 200-day SMA (a monthly panel alone would smear the trend filter
across ~21 trading days of resolution). The monthly panel — last regular-session close of each
month per ticker — is used for the momentum signal and for realized holding-period returns, same
as the pure momentum notebook.

In [ ]:
daily_closes = {}
for ticker in UNIVERSE:
    cache_path = DATA_DIR / f'{ticker}.csv'
    if not cache_path.exists():
        continue
    bars = load_cached_bars(ticker)
    if bars.empty:
        continue
    daily_closes[ticker] = bars['close']

price_panel = pd.DataFrame(daily_closes).sort_index()
price_panel = price_panel.loc[price_panel.index >= START_DATE]
monthly_prices = price_panel.resample('ME').last()
monthly_returns = monthly_prices.pct_change()

sma200_daily = price_panel.rolling(SMA_WINDOW_DAYS, min_periods=SMA_WINDOW_DAYS).mean()
sma200_monthly = sma200_daily.resample('ME').last().reindex(index=monthly_prices.index, columns=monthly_prices.columns)

print(f'{monthly_prices.shape[1]} tickers with cached data, {monthly_prices.shape[0]} months '
      f'from {monthly_prices.index.min().date()} to {monthly_prices.index.max().date()}')

## Signal, trend filter, and portfolio construction

At the rebalance point for month *t* (the close of month *t-1*):

1. **Momentum score** — the return from month *t-1-SKIP-LOOKBACK* to month *t-1-SKIP*: a 12-month
   return ending one month before the rebalance, exactly as in the pure momentum notebook.
2. **Momentum shortlist** — the top 30% of the eligible universe by that score.
3. **Trend filter** — within the shortlist, keep only names whose month *t-1* close is above their
   own 200-day SMA as of month *t-1* (no skip here: the trend filter uses the most current data
   available at the rebalance, unlike the deliberately-lagged momentum score).
4. **Selection** — the top 15 names by momentum rank among the names that pass the trend filter.
   Each of the 15 slots is worth 1/15 of the portfolio; an unfilled slot (too few names pass, or
   the universe itself is small that month) earns 0% for the month rather than being redistributed
   to the names that did qualify.

In [ ]:
dates = monthly_prices.index
portfolio_returns = pd.Series(index=dates, dtype=float)
holdings_log = {}

min_i = LOOKBACK_MONTHS + SKIP_MONTHS + 1
for i in range(min_i, len(dates)):
    hold_date = dates[i]
    decision_idx = i - 1
    end_idx = decision_idx - SKIP_MONTHS
    start_idx = end_idx - LOOKBACK_MONTHS
    momentum_score = (monthly_prices.iloc[end_idx] / monthly_prices.iloc[start_idx] - 1).dropna()

    if len(momentum_score) < MIN_UNIVERSE:
        holdings_log[hold_date] = []
        portfolio_returns.loc[hold_date] = 0.0
        continue

    ranked = momentum_score.sort_values(ascending=False)
    momentum_cutoff = max(1, int(np.ceil(len(ranked) * MOMENTUM_TOP_PCT)))
    momentum_candidates = ranked.index[:momentum_cutoff]

    price_now = monthly_prices.iloc[decision_idx]
    sma_now = sma200_monthly.iloc[decision_idx]
    trend_ok = (price_now > sma_now) & price_now.notna() & sma_now.notna()

    passing = ranked.loc[[t for t in momentum_candidates if trend_ok.get(t, False)]]
    selected = passing.sort_values(ascending=False).head(TOP_N).index.tolist()

    holdings_log[hold_date] = selected
    realized = monthly_returns.loc[hold_date, selected].fillna(0.0) if selected else pd.Series(dtype=float)
    portfolio_returns.loc[hold_date] = realized.sum() / TOP_N

invested_fraction = pd.Series({d: len(h) / TOP_N for d, h in holdings_log.items()})
print(f'{len(portfolio_returns)} months in the backtest, average {invested_fraction.mean():.0%} of '
      f'the target 15 slots filled, {(invested_fraction == 0).mean():.0%} of months fully in cash')

### Latest rebalance: example ranking

Same shape as the pure momentum notebook's example table, with the trend filter made explicit —
`in_momentum_top30`, `above_sma200`, and `selected` show exactly why each name did or didn't make
the final 15.

In [ ]:
last_hold_date = max(holdings_log)
last_decision_idx = list(dates).index(last_hold_date) - 1
last_end_idx = last_decision_idx - SKIP_MONTHS
last_start_idx = last_end_idx - LOOKBACK_MONTHS
last_score = (monthly_prices.iloc[last_end_idx] / monthly_prices.iloc[last_start_idx] - 1).dropna()
last_ranked = last_score.sort_values(ascending=False)
last_cutoff = max(1, int(np.ceil(len(last_ranked) * MOMENTUM_TOP_PCT)))

last_price = monthly_prices.iloc[last_decision_idx]
last_sma = sma200_monthly.iloc[last_decision_idx]

example_table = last_ranked.to_frame('trailing_12m_ex_1m_return')
example_table['rank'] = range(1, len(example_table) + 1)
example_table['in_momentum_top30'] = example_table.index.isin(last_ranked.index[:last_cutoff])
example_table['price'] = last_price.reindex(example_table.index)
example_table['sma200'] = last_sma.reindex(example_table.index)
example_table['above_sma200'] = example_table['price'] > example_table['sma200']
example_table['selected'] = example_table.index.isin(holdings_log[last_hold_date])
example_table['trailing_12m_ex_1m_return'] = (example_table['trailing_12m_ex_1m_return'] * 100).round(1)
example_table[['price', 'sma200']] = example_table[['price', 'sma200']].round(2)
print(f'Ranking and filters used to form the {last_hold_date.date()} holding month:')
display(example_table[example_table['in_momentum_top30']].head(30))

## General statistics

Same shape of summary as the other two notebooks, computed from this strategy's monthly return
series (which includes 0% months for unfilled/cash slots).

In [ ]:
def max_drawdown(wealth):
    drawdown = wealth / wealth.cummax() - 1.0
    return float(drawdown.min())

wealth = (1 + portfolio_returns).cumprod()
start_date, end_date = portfolio_returns.index[0], portfolio_returns.index[-1]
years = (end_date - start_date).days / 365.2425
total_return = float(wealth.iloc[-1] - 1.0)
cagr = float(wealth.iloc[-1] ** (1 / years) - 1.0)
annualized_volatility = float(portfolio_returns.std() * np.sqrt(12))
sharpe_rf0 = float((portfolio_returns.mean() * 12) / annualized_volatility) if annualized_volatility else float('nan')

stats = pd.Series({
    'observations_months': len(portfolio_returns),
    'start': start_date,
    'end': end_date,
    'avg_slots_filled_of_15': float((invested_fraction * TOP_N).mean()),
    'pct_months_fully_in_cash': float((invested_fraction == 0).mean()),
    'mean_monthly_return': float(portfolio_returns.mean()),
    'median_monthly_return': float(portfolio_returns.median()),
    'total_return': total_return,
    'CAGR': cagr,
    'annualized_volatility': annualized_volatility,
    'sharpe_ratio_rf0': sharpe_rf0,
    'max_drawdown': max_drawdown(wealth),
    'final_wealth_multiple': float(wealth.iloc[-1]),
})
display(stats.to_frame('value'))

## Buy-and-hold-style dollar simulation

Same purchasing-power framing as the other two notebooks: size a 2009-era investment so it has the
same purchasing power as $150 today (via FRED CPI), run it through this strategy's compounded
return (including its cash months), and compare the ending value against today's $150 in real
terms. Price-only, no-cost simulation: commissions, taxes, slippage, dividends, and the cash
sleeve's foregone interest are all excluded.

In [ ]:
try:
    from pandas_datareader import data as web
    cpi = web.DataReader('CPIAUCSL', 'fred', '2008-12-01', pd.Timestamp.now())['CPIAUCSL'].dropna()
    cpi_2009 = float(cpi.loc[cpi.index >= '2009-01-01'].iloc[0])
    target_cpi = float(TARGET_CPI) if TARGET_CPI is not None else float(cpi.iloc[-1])
    cpi_source = 'FRED CPIAUCSL'
except Exception as exc:
    if TARGET_CPI is None:
        raise RuntimeError('FRED CPI retrieval failed. Set TARGET_CPI to a CPIAUCSL value for the target dollar period.') from exc
    cpi_2009 = 211.143
    target_cpi = float(TARGET_CPI)
    cpi_source = 'manual target CPI; Jan 2009 CPI fallback'

initial_2009_dollars = INVESTMENT_2026_DOLLARS * cpi_2009 / target_cpi
ending_value = initial_2009_dollars * wealth.iloc[-1]
nominal_gain = ending_value - initial_2009_dollars
real_gain_vs_target_dollars = ending_value - INVESTMENT_2026_DOLLARS
real_multiple_vs_target_dollars = ending_value / INVESTMENT_2026_DOLLARS

simulation = pd.Series({
    'target_dollars': INVESTMENT_2026_DOLLARS,
    'cpi_2009': cpi_2009,
    'target_cpi': target_cpi,
    'initial_investment_in_2009_dollars': initial_2009_dollars,
    'ending_nominal_value': ending_value,
    'nominal_gain': nominal_gain,
    'total_return': total_return,
    'CAGR': cagr,
    'ending_value_in_target_dollars': ending_value,
    'real_gain_vs_target_dollars': real_gain_vs_target_dollars,
    'real_multiple_vs_target_dollars': real_multiple_vs_target_dollars,
})
print(f'CPI source: {cpi_source}')
display(simulation.to_frame('value'))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
wealth.plot(ax=ax, color='#176b87', title='Growth of $1 — 12-1 momentum + 200-day SMA trend filter, halal universe')
ax.set_ylabel('Portfolio value')
ax.axhline(1, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(13, 3))
(invested_fraction * 100).plot(ax=ax, color='#d97706', title='Portfolio slots filled (% of target 15)')
ax.set_ylabel('% of slots filled')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## Monthly performance

Monthly returns of the momentum-plus-trend-filter strategy, including months spent partly or
fully in cash.

In [ ]:
table = (portfolio_returns * 100).rename('ret').to_frame()
table['year'] = table.index.year
table['month'] = table.index.strftime('%b')
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_table = table.pivot(index='year', columns='month', values='ret').reindex(columns=month_order)

annual_return = portfolio_returns.groupby(portfolio_returns.index.year).apply(lambda x: (1 + x).prod() - 1) * 100
monthly_table['Year'] = annual_return

monthly_summary = pd.Series({
    'months': len(portfolio_returns),
    'positive_months': int((portfolio_returns > 0).sum()),
    'pct_positive_months': float((portfolio_returns > 0).mean()),
    'avg_monthly_return': float(portfolio_returns.mean()),
    'median_monthly_return': float(portfolio_returns.median()),
    'best_month': portfolio_returns.idxmax().strftime('%Y-%m'),
    'best_month_return': float(portfolio_returns.max()),
    'worst_month': portfolio_returns.idxmin().strftime('%Y-%m'),
    'worst_month_return': float(portfolio_returns.min()),
})

styled = (
    monthly_table.round(1)
    .style.format(precision=1, na_rep='')
    .background_gradient(cmap='RdYlGn', axis=None, vmin=-15, vmax=15, subset=month_order)
    .set_caption('Monthly return (%) by year')
)
display(styled)
display(monthly_summary.to_frame('value'))

In [ ]:
colors = np.where(portfolio_returns >= 0, '#176b87', '#c0392b')
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(portfolio_returns.index, portfolio_returns.values * 100, width=20, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Momentum + trend filter monthly return (%)')
ax.set_ylabel('Return (%)')
plt.tight_layout()
plt.show()

## Caveats and next improvements

- **Look-ahead / survivorship bias in the universe**: `equities-list.csv` is today's list of halal
  candidates, projected backward. Stocks that were later added to the universe (or that would have
  been screened out earlier) are not reconstructed historically, and delisted/renamed names are not
  included at all. A rigorous backtest needs a *point-in-time* halal-screened universe.
- **Price-only bars**: like the other notebooks, this uses IBKR `TRADES` bars — split-adjusted but
  not dividend-adjusted.
- **No transaction costs**: commissions, spread, and market impact from monthly rebalancing across
  up to 15 names are excluded. The trend filter adds extra turnover on top of the pure momentum
  version (names get sold on a trend break even if momentum alone would have kept them), so real
  costs would bite this version somewhat harder.
- **Cash earns 0%, not a risk-free rate**: unfilled slots contribute nothing to the return series.
  A more realistic version would credit cash with T-bill-like interest, which matters in this
  strategy specifically because it can spend real time out of the market during broad drawdowns.
- **SMA filter uses un-lagged, un-skipped data**: the momentum score deliberately skips the most
  recent month; the trend filter deliberately does not, since a trend filter is meant to reflect
  the most current state of the trend. That asymmetry is intentional but worth knowing about if you
  extend this.
- **Ranking universe drift**: many tickers (recent IPOs, in particular) don't have 13 months of
  history in earlier years and simply aren't eligible until they do; `MIN_UNIVERSE` guards against
  ranking with too few names but doesn't correct for the universe being thin in its earliest years.
- A serious version of this strategy would need a point-in-time halal screen, total-return prices,
  realistic transaction costs and cash yield, and probably a comparison against the pure momentum
  notebook to quantify what the trend filter actually bought you (higher risk-adjusted return? or
  just fewer, better-timed drawdowns at the cost of some upside?).